In [1]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/functions/across_cohort_descriptive_func.R')
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/functions/ind_descriptive_func.R')

path.in <- '/home/jupyter/workspaces/result/final/'
path.out <- '/home/jupyter/workspaces/result/final/'

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor



Attaching package: ‘rstatix’


The following object is masked from ‘package:stats’:

    filter


corrplot 0.95 loaded


Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The 

In [2]:
final_pid <- read.delim(
    paste0(path.in, 'final_pid.txt'),
    as.is = T
);

dim(final_pid)

[1] 9312    2

In [3]:
# fitbit activity
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Diabetes'),'person_id'], collapse = ", ")
diabetes_activity_sql <- paste("
       SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary  
    WHERE
        activity_summary.PERSON_ID IN (", diabetes.pid.sql, ")"
)

diabetes_activity_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_activity",
  "diabetes_activity_*.csv")
message(str_glue('The data will be written to {diabetes_activity_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_activity_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_activity/diabetes_activity_*.csv. Use this path when reading the data into your notebooks in the future.



In [4]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_activity_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_activity/diabetes_activity_*.csv'

diabetes_activity <- read_bq_export_from_workspace_bucket(diabetes_activity_path)

dim(diabetes_activity)

head(diabetes_activity, 2) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_activity/diabetes_activity_000000000000.csv.



[1] 489116     13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
4196048,2023-09-30,1821,2295,3970,NA,52,NA,229,1097,479,2976,12
4196048,2023-07-21,791,2390,3309,NA,0,NA,153,393,782,1809,0


In [5]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_meta/diabetes_meta_*.csv'

diabetes_meta <- read_bq_export_from_workspace_bucket(diabetes_meta_path)

dim(diabetes_meta)

head(diabetes_meta, 3) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_meta/diabetes_meta_000000000000.csv.



[1] 1298   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1972705,1963-06-15 00:00:00 UTC,1177221,I prefer not to answer,903079,PMI: Prefer Not To Answer,45878463,Female,1177221,I prefer not to answer
8423567,1978-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,45882607,None of these
2419350,1993-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,45882607,None of these


In [6]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Diabetes/', df = diabetes_activity)

── Wear compliance ──
     n_days       followup_days   
 Min.   :  25.0   Min.   :  25.0  
 1st Qu.:  97.0   1st Qu.:  98.0  
 Median : 112.5   Median : 114.5  
 Mean   : 376.8   Mean   : 382.4  
 3rd Qu.: 443.8   3rd Qu.: 445.8  
 Max.   :3898.0   Max.   :3967.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1003770   Min.   :  25.0   Min.   :2012-11-21   Min.   :2018-07-09  
 1st Qu.:1940734   1st Qu.:  97.0   1st Qu.:2022-05-01   1st Qu.:2023-10-01  
 Median :3361450   Median : 112.5   Median :2023-06-08   Median :2023-10-01  
 Mean   :4321604   Mean   : 376.8   Mean   :2022-07-21   Mean   :2023-08-06  
 3rd Qu.:6542119   3rd Qu.: 443.8   3rd Qu.:2023-06-24   3rd Qu.:2023-10-01  
 Max.   :9987610   Max.   :3898.0   Max.   :2023-09-07   Max.   :2023-10-01  
 followup_days     missing_days      pct_missing        compliance_tier
 Min.   :  25.0   Min.   :  0.000   Min.   : 0.000   High (>90%):1254  
 1st Qu.:  98.0   1st Qu.:  0.000   1st Qu.: 0.000   80-90%     :  41  
 Median : 114.5   Median :  0.000   Median : 0.000   70-80%     :   3  
 Mean   : 382.4   Mean   :  5.609   Mean   : 1.544   60-70%     :   0  
 3rd Qu.: 445.8   3rd Qu.:  3.000   3rd Qu.: 1.796   50-60%     :   0  
 Max.   :3967.0   Max.

NULL

In [7]:
miss_gap_func(df = diabetes_activity, path.out =  '/home/jupyter/workspaces/result/final/Diabetes/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   0.00    0.00    0.00    1.69    1.00   14.00 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0        512        39.4         786         60.6
2         7        106         8.17       1192         91.8
3        14          0         0          1298        100  
4        30          0         0          1298        100  
5        60          0         0          1298        100  


NULL

In [8]:
diabetes_patient_stats <- activity_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Diabetes/',
    cohort = 'Diabetes', df = diabetes_activity,
    meta.data = diabetes_meta[,c('person_id', 'sex_at_birth')])

# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out            1298 2337.   572.  2.28e3 1.92e3 2664.  1.53e+3 3.42e3
2 fairly_active_minutes   1298   12.2   12.2 8.65e0 3.36e0   17.0 5.00e-1 3.60e1
3 lightly_active_minutes  1298  177.    73.4 1.74e2 1.26e2  224.  5.73e+1 3.00e2
4 sedentary_minutes       1298  825.   150.  8.06e2 7.28e2  888.  6.27e+2 1.12e3
5 steps                   1298 5267.  3348.  4.68e3 2.80e3 7051.  1.05e+3 1.16e4
6 very_active_minutes     1298   12.1   16.4 5.72e0 1.68e0   15.8 2.24e-1 4.75e1


In [9]:
# fitbit activity
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Insomnia'),'person_id'], collapse = ", ")
insomnia_activity_sql <- paste("
       SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary  
    WHERE
        activity_summary.PERSON_ID IN (", insomnia.pid.sql, ")"
)

insomnia_activity_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_activity",
  "insomnia_activity_*.csv")
message(str_glue('The data will be written to {insomnia_activity_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_activity_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_activity/insomnia_activity_*.csv. Use this path when reading the data into your notebooks in the future.



In [10]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_activity_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_activity/insomnia_activity_*.csv'

insomnia_activity <- read_bq_export_from_workspace_bucket(insomnia_activity_path)

dim(insomnia_activity)

head(insomnia_activity, 2) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_activity/insomnia_activity_000000000000.csv.



[1] 567701     13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1635625,2023-06-03,37,1521,1543,NA,0,NA,14,13,1426,110,0
1635625,2023-05-31,868,1521,2033,NA,0,NA,343,315,1097,1450,0


In [11]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_meta/insomnia_meta_*.csv'

insomnia_meta <- read_bq_export_from_workspace_bucket(insomnia_meta_path)

dim(insomnia_meta)

head(insomnia_meta, 2) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_meta/insomnia_meta_000000000000.csv.



[1] 1383   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
2105443,1959-06-15 00:00:00 UTC,8557,Native Hawaiian or Other Pacific Islander,38003564,Not Hispanic or Latino,45880669,Male,8557,Native Hawaiian or Other Pacific Islander
6725688,1948-06-15 00:00:00 UTC,8557,Native Hawaiian or Other Pacific Islander,38003564,Not Hispanic or Latino,45878463,Female,8557,Native Hawaiian or Other Pacific Islander


In [12]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Insomnia/', df = insomnia_activity)

── Wear compliance ──
     n_days       followup_days   
 Min.   :  23.0   Min.   :  23.0  
 1st Qu.:  96.0   1st Qu.:  97.0  
 Median : 112.0   Median : 112.0  
 Mean   : 410.5   Mean   : 417.3  
 3rd Qu.: 467.0   3rd Qu.: 469.0  
 Max.   :5386.0   Max.   :5386.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1003126   Min.   :  23.0   Min.   :2009-01-02   Min.   :2017-11-21  
 1st Qu.:1870796   1st Qu.:  96.0   1st Qu.:2022-03-05   1st Qu.:2023-10-01  
 Median :3149420   Median : 112.0   Median :2023-06-10   Median :2023-10-01  
 Mean   :4182739   Mean   : 410.5   Mean   :2022-06-09   Mean   :2023-07-30  
 3rd Qu.:6451714   3rd Qu.: 467.0   3rd Qu.:2023-06-25   3rd Qu.:2023-10-01  
 Max.   :9987114   Max.   :5386.0   Max.   :2023-09-09   Max.   :2023-10-01  
 followup_days     missing_days      pct_missing        compliance_tier
 Min.   :  23.0   Min.   :  0.000   Min.   : 0.000   High (>90%):1344  
 1st Qu.:  97.0   1st Qu.:  0.000   1st Qu.: 0.000   80-90%     :  37  
 Median : 112.0   Median :  0.000   Median : 0.000   70-80%     :   2  
 Mean   : 417.3   Mean   :  6.797   Mean   : 1.591   60-70%     :   0  
 3rd Qu.: 469.0   3rd Qu.:  3.000   3rd Qu.: 1.887   50-60%     :   0  
 Max.   :5386.0   Max.

NULL

In [13]:
miss_gap_func(df = insomnia_activity, path.out =  '/home/jupyter/workspaces/result/final/Insomnia/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.000   0.000   1.864   2.000  14.000 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0        589        42.6         794         57.4
2         7        125         9.04       1258         91.0
3        14          0         0          1383        100  
4        30          0         0          1383        100  
5        60          0         0          1383        100  


NULL

In [14]:
insomnia_patient_stats <- activity_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Insomnia/',
    cohort = 'Insomnia', df = insomnia_activity,
    meta.data = insomnia_meta[,c('person_id', 'sex_at_birth')])

# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out            1383 2210.   566.  2.14e3 1.78e3 2552.  1.43e+3 3.21e3
2 fairly_active_minutes   1383   13.7   13.3 9.95e0 4.47e0   19.0 8.32e-1 3.94e1
3 lightly_active_minutes  1383  189.    74.6 1.91e2 1.41e2  234.  6.25e+1 3.09e2
4 sedentary_minutes       1383  795.   155.  7.75e2 7.01e2  864.  5.86e+2 1.09e3
5 steps                   1383 5894.  3436.  5.47e3 3.30e3 7798.  1.24e+3 1.23e4
6 very_active_minutes     1383   13.7   16.9 7.41e0 2.53e0   18.7 3.39e-1 4.69e1


In [15]:
# fitbit activity
library(tidyverse)
library(bigrquery)

pca.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Prostate'),'person_id'], collapse = ", ")
pca_activity_sql <- paste("
       SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary  
    WHERE
        activity_summary.PERSON_ID IN (", pca.pid.sql, ")"
)

pca_activity_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "pca_activity",
  "pca_activity_*.csv")
message(str_glue('The data will be written to {pca_activity_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), pca_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  pca_activity_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_activity/pca_activity_*.csv. Use this path when reading the data into your notebooks in the future.



In [16]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_activity_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_activity/pca_activity_*.csv'

pca_activity <- read_bq_export_from_workspace_bucket(pca_activity_path)

dim(pca_activity)

head(pca_activity, 2) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_activity/pca_activity_000000000000.csv.



[1] 100358     13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2967502,2023-09-21,202,1545,1723,NA,0,NA,55,97,1385,1320,0
2967502,2023-07-26,597,1568,2086,NA,0,NA,170,277,679,3413,0


In [17]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_meta/pca_meta_*.csv'

pca_meta <- read_bq_export_from_workspace_bucket(pca_meta_path)

dim(pca_meta)

head(pca_meta, 2)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_meta/pca_meta_000000000000.csv.



[1] 250  10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
2967502,1953-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,45882607,None of these
2462406,1948-06-15 00:00:00 UTC,8527,White,38003563,Hispanic or Latino,45880669,Male,2000000008,More than one population


In [18]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Prostate/', df = pca_activity)

── Wear compliance ──
     n_days       followup_days    
 Min.   :  24.0   Min.   :  24.00  
 1st Qu.:  93.0   1st Qu.:  94.25  
 Median : 106.0   Median : 107.00  
 Mean   : 401.4   Mean   : 408.36  
 3rd Qu.: 272.5   3rd Qu.: 276.00  
 Max.   :3491.0   Max.   :3524.00  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1015411   Min.   :  24.0   Min.   :2014-01-17   Min.   :2020-06-21  
 1st Qu.:1903888   1st Qu.:  93.0   1st Qu.:2022-12-22   1st Qu.:2023-10-01  
 Median :3428482   Median : 106.0   Median :2023-06-16   Median :2023-10-01  
 Mean   :4265862   Mean   : 401.4   Mean   :2022-07-30   Mean   :2023-09-10  
 3rd Qu.:6505875   3rd Qu.: 272.5   3rd Qu.:2023-06-28   3rd Qu.:2023-10-01  
 Max.   :9933104   Max.   :3491.0   Max.   :2023-09-08   Max.   :2023-10-01  
 followup_days      missing_days      pct_missing        compliance_tier
 Min.   :  24.00   Min.   :  0.000   Min.   : 0.000   High (>90%):241   
 1st Qu.:  94.25   1st Qu.:  0.000   1st Qu.: 0.000   80-90%     :  9   
 Median : 107.00   Median :  0.000   Median : 0.000   70-80%     :  0   
 Mean   : 408.36   Mean   :  6.928   Mean   : 1.325   60-70%     :  0   
 3rd Qu.: 276.00   3rd Qu.:  2.000   3rd Qu.: 1.143   50-60%     :  0   
 Max.   :3524.00

NULL

In [19]:
miss_gap_func(df = pca_activity, path.out =  '/home/jupyter/workspaces/result/final/Prostate/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.000   0.000   1.688   1.000  14.000 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0         91         36.4        159         63.6
2         7         22          8.8        228         91.2
3        14          0          0          250        100  
4        30          0          0          250        100  
5        60          0          0          250        100  


NULL

In [20]:
pca_patient_stats <- activity_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Prostate/',
    cohort = 'Prostate', df = pca_activity,
    meta.data = pca_meta[,c('person_id', 'sex_at_birth')])

# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out             250 2521.   417.  2509.  2.23e3 2774.  1.92e+3 3.27e3
2 fairly_active_minutes    250   19.5   16.5   17.6 6.75e0   27.3 1.64e+0 5.08e1
3 lightly_active_minutes   250  185.    62.7  191.  1.45e2  227.  7.80e+1 2.81e2
4 sedentary_minutes        250  780.   132.   767.  6.94e2  851.  5.98e+2 1.02e3
5 steps                    250 7066.  3922.  6614.  4.49e3 9226.  1.55e+3 1.26e4
6 very_active_minutes      250   23.2   22.3   17.0 5.62e0   30.8 5.49e-1 6.63e1


In [21]:
# fitbit activity
library(tidyverse)
library(bigrquery)

healthy.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Healthy'),'person_id'], collapse = ", ")
healthy_activity_sql <- paste("
       SELECT
        activity_summary.person_id,
        activity_summary.date,
        activity_summary.activity_calories,
        activity_summary.calories_bmr,
        activity_summary.calories_out,
        activity_summary.elevation,
        activity_summary.fairly_active_minutes,
        activity_summary.floors,
        activity_summary.lightly_active_minutes,
        activity_summary.marginal_calories,
        activity_summary.sedentary_minutes,
        activity_summary.steps,
        activity_summary.very_active_minutes 
    FROM
        `activity_summary` activity_summary  
    WHERE
        activity_summary.PERSON_ID IN (", healthy.pid.sql, ")"
)

healthy_activity_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "healthy_activity",
  "healthy_activity_*.csv")
message(str_glue('The data will be written to {healthy_activity_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), healthy_activity_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  healthy_activity_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_activity/healthy_activity_*.csv. Use this path when reading the data into your notebooks in the future.



In [22]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

healthy_activity_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_activity/healthy_activity_*.csv'

healthy_activity <- read_bq_export_from_workspace_bucket(healthy_activity_path)

dim(healthy_activity)

head(healthy_activity, 2)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_activity/healthy_activity_000000000000.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_activity/healthy_activity_000000000001.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_activity/healthy_activity_000000000002.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_activity/healthy_activity_000000000003.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_activity/healthy_activity_000000000004.csv.

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_activity/healthy_activity_000000000005.csv.



[1] 3256757      13

person_id,date,activity_calories,calories_bmr,calories_out,elevation,fairly_active_minutes,floors,lightly_active_minutes,marginal_calories,sedentary_minutes,steps,very_active_minutes
<dbl>,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1951790,2023-08-10,1199,1480,2538,NA,11,NA,134,808,773,14642,79
1951790,2023-06-21,1348,1480,2685,NA,40,NA,73,948,647,18597,119


In [23]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_meta/healthy_meta_*.csv'

healthy_meta <- read_bq_export_from_workspace_bucket(healthy_meta_path)

dim(healthy_meta)

head(healthy_meta, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_meta/healthy_meta_000000000000.csv.



[1] 6381   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
7243587,1966-06-15 00:00:00 UTC,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,1585849,Sex At Birth: Sex At Birth None Of These,8657,American Indian or Alaska Native
3023133,1981-06-15 00:00:00 UTC,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,45878463,Female,8657,American Indian or Alaska Native
5914245,1977-06-15 00:00:00 UTC,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,45878463,Female,8657,American Indian or Alaska Native
6241747,1953-06-15 00:00:00 UTC,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,45878463,Female,8657,American Indian or Alaska Native
9584907,1955-06-15 00:00:00 UTC,8657,American Indian or Alaska Native,38003564,Not Hispanic or Latino,45878463,Female,8657,American Indian or Alaska Native


In [24]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Healthy/', df = healthy_activity)

── Wear compliance ──
     n_days       followup_days 
 Min.   :  22.0   Min.   :  22  
 1st Qu.: 101.0   1st Qu.: 102  
 Median : 147.0   Median : 151  
 Mean   : 510.4   Mean   : 517  
 3rd Qu.: 736.0   3rd Qu.: 741  
 Max.   :4154.0   Max.   :4198  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1000464   Min.   :  22.0   Min.   :2012-04-04   Min.   :2016-02-05  
 1st Qu.:2251610   1st Qu.: 101.0   1st Qu.:2021-06-24   1st Qu.:2023-10-01  
 Median :4194375   Median : 147.0   Median :2023-03-29   Median :2023-10-01  
 Mean   :4719200   Mean   : 510.4   Mean   :2022-03-01   Mean   :2023-07-30  
 3rd Qu.:7058659   3rd Qu.: 736.0   3rd Qu.:2023-06-20   3rd Qu.:2023-10-01  
 Max.   :9997301   Max.   :4154.0   Max.   :2023-09-09   Max.   :2023-10-01  
 followup_days   missing_days      pct_missing        compliance_tier
 Min.   :  22   Min.   :  0.000   Min.   : 0.000   High (>90%):6202  
 1st Qu.: 102   1st Qu.:  0.000   1st Qu.: 0.000   80-90%     : 161  
 Median : 151   Median :  0.000   Median : 0.000   70-80%     :  18  
 Mean   : 517   Mean   :  6.635   Mean   : 1.408   60-70%     :   0  
 3rd Qu.: 741   3rd Qu.:  3.000   3rd Qu.: 1.515   50-60%     :   0  
 Max.   :4198   Max.   :381.000   

NULL

In [25]:
miss_gap_func(df = healthy_activity, path.out =  '/home/jupyter/workspaces/result/final/Healthy/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.000   0.000   2.043   2.000  14.000 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0       2465         38.6       3916         61.4
2         7        740         11.6       5641         88.4
3        14          0          0         6381        100  
4        30          0          0         6381        100  
5        60          0          0         6381        100  


NULL

In [26]:
healthy_patient_stats <- activity_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Healthy/',
    cohort = 'Healthy', df = healthy_activity,
    meta.data = healthy_meta[,c('person_id', 'sex_at_birth')])

# A tibble: 6 × 9
  metric                     N   Mean     SD Median     Q1     Q3      P5    P95
  <chr>                  <int>  <dbl>  <dbl>  <dbl>  <dbl>  <dbl>   <dbl>  <dbl>
1 calories_out            6381 2283.   589.  2197.  1.85e3 2646.  1.47e+3 3.35e3
2 fairly_active_minutes   6381   16.5   15.3   12.5 6.23e0   22.3 1.25e+0 4.48e1
3 lightly_active_minutes  6381  200.    78.2  199.  1.50e2  249.  6.28e+1 3.28e2
4 sedentary_minutes       6381  785.   173.   756.  6.78e2  847.  5.72e+2 1.18e3
5 steps                   6381 6952.  3998.  6452.  4.09e3 9188.  1.45e+3 1.38e4
6 very_active_minutes     6381   18.9   21.6   11.8 4.16e0   26.6 4.88e-1 5.93e1


In [27]:
### Across Cohort activity ################################
activity.sum.all <- rbind(
    diabetes_patient_stats,
    insomnia_patient_stats,
    pca_patient_stats,
    healthy_patient_stats);
activity.sum.all[1:4,]
ac_active_plot_func(df = activity.sum.all, path.out = paste0(path.out, 'AcrossCohort/activity/'))


,person_id,n_obs,steps__mean,steps__sd,steps__cv,calories_out__mean,calories_out__sd,calories_out__cv,sedentary_minutes__mean,sedentary_minutes__sd,⋯,lightly_active_minutes__sd,lightly_active_minutes__cv,fairly_active_minutes__mean,fairly_active_minutes__sd,fairly_active_minutes__cv,very_active_minutes__mean,very_active_minutes__sd,very_active_minutes__cv,cohort,sex_at_birth
,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,1003770,32,1564.969,1184.697,75.70101,1554.000,316.5075,20.367275,1050.9062,197.8588,⋯,58.83170,64.05629,15.250000,39.433079,258.5776,3.843750,12.869342,334.8121,Diabetes,Female
2,1005805,66,5801.136,2190.840,37.76571,2814.682,386.5767,13.734296,662.5152,103.6625,⋯,56.39954,22.13588,21.045455,24.238832,115.1737,13.621212,23.775725,174.5493,Diabetes,Female
3,1011367,110,6336.291,3117.370,49.19866,2156.891,241.9721,11.218560,728.7455,195.3681,⋯,85.99445,39.74199,9.781818,11.763247,120.2562,9.190909,14.333828,155.9566,Diabetes,Female
4,1016057,51,3229.824,1284.816,39.77975,1900.020,173.9891,9.157228,886.8235,267.8715,⋯,61.03792,38.58850,1.705882,4.300205,252.0810,0.254902,1.246485,489.0058,Diabetes,Female


`geom_smooth()` using formula = 'y ~ x'


NULL

In [28]:
#### sleep ########
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Diabetes'),'person_id'], collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
diabetes_sleep_sql <- paste("
    SELECT
        sleep_daily_summary.person_id,
        sleep_daily_summary.sleep_date,
        sleep_daily_summary.is_main_sleep,
        sleep_daily_summary.minute_in_bed,
        sleep_daily_summary.minute_asleep,
        sleep_daily_summary.minute_after_wakeup,
        sleep_daily_summary.minute_awake,
        sleep_daily_summary.minute_restless,
        sleep_daily_summary.minute_deep,
        sleep_daily_summary.minute_light,
        sleep_daily_summary.minute_rem,
        sleep_daily_summary.minute_wake 
    FROM
        `sleep_daily_summary` sleep_daily_summary   
    WHERE
        PERSON_ID IN (", diabetes.pid.sql, ")"
)
# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
diabetes_sleep_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_sleep",
  "diabetes_sleep.csv")
message(str_glue('The data will be written to {diabetes_sleep_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_sleep_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_sleep_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_sleep/diabetes_sleep.csv. Use this path when reading the data into your notebooks in the future.



In [29]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(is_main_sleep = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_sleep_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_sleep/diabetes_sleep.csv'

diabetes_sleep <- read_bq_export_from_workspace_bucket(diabetes_sleep_path)

dim(diabetes_sleep)

head(diabetes_sleep, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_sleep/diabetes_sleep.csv.



[1] 507607     12

person_id,sleep_date,is_main_sleep,minute_in_bed,minute_asleep,minute_after_wakeup,minute_awake,minute_restless,minute_deep,minute_light,minute_rem,minute_wake
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3349357,2022-12-10,false,182,161,0,21,NA,30,105,26,21
3349357,2022-06-06,true,182,159,8,23,NA,14,123,22,23
3352796,2023-09-11,false,183,174,0,9,NA,28,110,36,9
3289024,2023-06-18,true,183,168,0,15,NA,21,115,32,15
3289024,2023-08-20,false,183,160,0,23,NA,11,117,32,23


In [30]:
colnames(diabetes_sleep)[2] <- 'date'
diabetes_sleep_main <- diabetes_sleep |>
  filter(is_main_sleep == "true")

diabetes_sleep_nap <- diabetes_sleep |>
  filter(is_main_sleep == "false")

diabetes_sleep_sum <- sleep_sum_main_nap_func(df = diabetes_sleep)

In [31]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Diabetes/sleep/', df = diabetes_sleep_sum)

── Wear compliance ──
     n_days       followup_days   
 Min.   :  21.0   Min.   :  21.0  
 1st Qu.:  79.0   1st Qu.:  91.0  
 Median :  95.0   Median : 109.0  
 Mean   : 325.6   Mean   : 349.4  
 3rd Qu.: 273.0   3rd Qu.: 307.8  
 Max.   :3881.0   Max.   :3966.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1003770   Min.   :  21.0   Min.   :2012-11-22   Min.   :2018-07-09  
 1st Qu.:1940734   1st Qu.:  79.0   1st Qu.:2022-07-10   1st Qu.:2023-09-30  
 Median :3361450   Median :  95.0   Median :2023-06-13   Median :2023-10-01  
 Mean   :4321604   Mean   : 325.6   Mean   :2022-08-15   Mean   :2023-07-29  
 3rd Qu.:6542119   3rd Qu.: 273.0   3rd Qu.:2023-07-03   3rd Qu.:2023-10-01  
 Max.   :9987610   Max.   :3881.0   Max.   :2023-09-11   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing        compliance_tier
 Min.   :  21.0   Min.   :  0.00   Min.   : 0.000   High (>90%):778   
 1st Qu.:  91.0   1st Qu.:  5.00   1st Qu.: 3.297   80-90%     :332   
 Median : 109.0   Median : 13.00   Median : 7.470   70-80%     :188   
 Mean   : 349.4   Mean   : 23.78   Mean   : 9.844   60-70%     :  0   
 3rd Qu.: 307.8   3rd Qu.: 25.00   3rd Qu.:14.596   50-60%     :  0   
 Max.   :3966.0   Max.   :31

NULL

In [32]:
miss_gap_func(df = diabetes_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Diabetes/sleep/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   0.00    1.00    3.00    5.27    9.00   14.00 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0       1253         96.5         45         3.47
2         7        373         28.7        925        71.3 
3        14          0          0         1298       100   
4        30          0          0         1298       100   
5        60          0          0         1298       100   


NULL

In [33]:
patient_stats_main <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Diabetes/sleep/',
    cohort = 'Diabetes',
    df = diabetes_sleep_main,
    main = 'main'
)
patient_stats_nap <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Diabetes/sleep/',
    cohort = 'Diabetes',
    df = diabetes_sleep_nap,
    main = 'nap'
)

# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep  1298 370.   66.3  377.  331.  414.  246.  467. 
2 minute_awake   1298  52.6  13.7   51.9  43.5  61.0  30.4  74.1
3 minute_deep    1295  53.5  15.3   53.2  41.8  64.0  30.6  79.1
4 minute_in_bed  1298 423.   74.7  431.  380.  472.  289.  533. 
5 minute_light   1295 257.   44.7  254.  228.  285.  187.  333. 
6 minute_rem     1295  72.3  21.2   72.0  58.2  86.5  37.9 106. 
7 minute_wake    1295  55.4  12.0   54.5  47.5  63.2  37.6  75.2
# A tibble: 7 × 9
  metric            N  Mean    SD Median     Q1    Q3     P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl>  <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep  1285 103.  25.1  102.    88.2  117.   66.7  145. 
2 minute_awake   1285  12.3  9.02   9.62   6.68  14.1   3.84  32.9
3 minute_deep     976  24.0 11.4   24     16.4   30.7   6     42.5
4 minute_in_bed  1285 117.  26.8  115.    99

In [34]:
head(diabetes_sleep_sum)
patient_stats_sum <-sleep_prep_stats(df = diabetes_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Diabetes/sleep/', cohort = 'Diabetes', main = "sum")

person_id,date,has_main_sleep,has_nap,n_records,minute_in_bed,minute_asleep,minute_awake,minute_restless,minute_deep,⋯,minute_wake,minute_after_wakeup,main_minute_in_bed,main_minute_asleep,main_minute_awake,main_minute_deep,main_minute_light,main_minute_rem,main_minute_wake,sleep_efficiency
<dbl>,<date>,<lgl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1003770,2023-08-31,TRUE,FALSE,1,294,256,38,0,52,⋯,38,0,294,256,38,52,152,52,38,87.07483
1003770,2023-09-01,TRUE,FALSE,1,151,148,3,3,0,⋯,0,0,151,148,3,0,0,0,0,98.01325
1003770,2023-09-02,TRUE,FALSE,1,344,319,25,0,34,⋯,25,5,344,319,25,34,211,74,25,92.73256
1003770,2023-09-03,TRUE,TRUE,2,377,349,28,0,57,⋯,28,0,377,349,28,57,122,87,28,92.57294
1003770,2023-09-04,TRUE,FALSE,1,213,191,22,0,42,⋯,22,0,213,191,22,42,86,63,22,89.67136
1003770,2023-09-05,TRUE,FALSE,1,150,149,1,1,0,⋯,0,0,150,149,1,0,0,0,0,99.33333


# A tibble: 8 × 9
  metric               N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>            <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep     1298 391.  64.0   394.  355.  429.  282.  485. 
2 minute_awake      1298  54.9 14.2    54.1  45.5  64.0  32.7  78.5
3 minute_deep       1298  48.0 17.4    48.1  36.2  60.0  19.2  76.3
4 minute_in_bed     1298 447.  71.8   450.  406.  489.  329.  560. 
5 minute_light      1298 229.  63.0   236.  197.  270.  114.  320. 
6 minute_rem        1298  65.0 23.9    66.1  50.2  80.8  23.4 101. 
7 minute_wake       1298  49.6 15.1    50.2  41.0  59.3  24.0  72.1
8 sleep_efficiency  1298  87.6  3.11   88.0  86.5  89.4  83.1  91.3


In [35]:
#### sleep ########
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Insomnia'),'person_id'], collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
insomnia_sleep_sql <- paste("
    SELECT
        sleep_daily_summary.person_id,
        sleep_daily_summary.sleep_date,
        sleep_daily_summary.is_main_sleep,
        sleep_daily_summary.minute_in_bed,
        sleep_daily_summary.minute_asleep,
        sleep_daily_summary.minute_after_wakeup,
        sleep_daily_summary.minute_awake,
        sleep_daily_summary.minute_restless,
        sleep_daily_summary.minute_deep,
        sleep_daily_summary.minute_light,
        sleep_daily_summary.minute_rem,
        sleep_daily_summary.minute_wake 
    FROM
        `sleep_daily_summary` sleep_daily_summary   
    WHERE
        PERSON_ID IN (", insomnia.pid.sql, ")"
)
# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
insomnia_sleep_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_sleep",
  "insomnia_sleep.csv")
message(str_glue('The data will be written to {insomnia_sleep_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_sleep_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_sleep_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_sleep/insomnia_sleep.csv. Use this path when reading the data into your notebooks in the future.



In [36]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(is_main_sleep = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_sleep_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_sleep/insomnia_sleep.csv'

insomnia_sleep <- read_bq_export_from_workspace_bucket(insomnia_sleep_path)

dim(insomnia_sleep)

head(insomnia_sleep, 2)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_sleep/insomnia_sleep.csv.



[1] 573761     12

person_id,sleep_date,is_main_sleep,minute_in_bed,minute_asleep,minute_after_wakeup,minute_awake,minute_restless,minute_deep,minute_light,minute_rem,minute_wake
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3384650,2023-04-06,true,186,172,6,14,NA,13,149,10,14
3377521,2023-03-08,true,186,166,0,20,NA,29,119,18,20


In [37]:
colnames(insomnia_sleep)[2] <- 'date'
insomnia_sleep_main <- insomnia_sleep |>
  filter(is_main_sleep == "true")

insomnia_sleep_nap <- insomnia_sleep |>
  filter(is_main_sleep == "false")

insomnia_sleep_sum <- sleep_sum_main_nap_func(df = insomnia_sleep)

In [38]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Insomnia/sleep/', df = insomnia_sleep_sum)

── Wear compliance ──
     n_days       followup_days   
 Min.   :  21.0   Min.   :  21.0  
 1st Qu.:  77.0   1st Qu.:  90.0  
 Median :  95.0   Median : 108.0  
 Mean   : 352.6   Mean   : 376.9  
 3rd Qu.: 332.5   3rd Qu.: 372.5  
 Max.   :4166.0   Max.   :4364.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1003126   Min.   :  21.0   Min.   :2011-09-30   Min.   :2017-11-20  
 1st Qu.:1870796   1st Qu.:  77.0   1st Qu.:2022-06-03   1st Qu.:2023-09-30  
 Median :3149420   Median :  95.0   Median :2023-06-13   Median :2023-10-01  
 Mean   :4182739   Mean   : 352.6   Mean   :2022-07-12   Mean   :2023-07-23  
 3rd Qu.:6451714   3rd Qu.: 332.5   3rd Qu.:2023-07-03   3rd Qu.:2023-10-01  
 Max.   :9987114   Max.   :4166.0   Max.   :2023-09-11   Max.   :2023-10-01  
 followup_days     missing_days    pct_missing        compliance_tier
 Min.   :  21.0   Min.   :  0.0   Min.   : 0.000   High (>90%):862   
 1st Qu.:  90.0   1st Qu.:  4.0   1st Qu.: 3.137   80-90%     :347   
 Median : 108.0   Median : 12.0   Median : 7.235   70-80%     :174   
 Mean   : 376.9   Mean   : 24.3   Mean   : 9.341   60-70%     :  0   
 3rd Qu.: 372.5   3rd Qu.: 25.0   3rd Qu.:14.145   50-60%     :  0   
 Max.   :4364.0   Max.   :316.0   

NULL

In [39]:
miss_gap_func(df = insomnia_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Insomnia/sleep/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   1.000   3.000   5.206   9.000  14.000 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0       1340         96.9         43         3.11
2         7        397         28.7        986        71.3 
3        14          0          0         1383       100   
4        30          0          0         1383       100   
5        60          0          0         1383       100   


NULL

In [42]:
patient_stats_main <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Insomnia/sleep/',
    cohort = 'Insomnia',
    df = insomnia_sleep_main,
    main = 'main'
)
patient_stats_nap <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Insomnia/sleep/',
    cohort = 'Insomnia',
    df = insomnia_sleep_nap,
    main = 'nap'
)

# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep  1383 382.   69.6  390.  343.  427.  255.  484. 
2 minute_awake   1383  54.5  14.4   54.1  45.7  62.5  33.5  78.2
3 minute_deep    1382  56.9  16.2   57.1  44.7  68.2  31.0  83.7
4 minute_in_bed  1383 437.   77.6  446.  391.  486.  298.  549. 
5 minute_light   1382 261.   44.9  260.  230.  290.  190.  333. 
6 minute_rem     1382  75.7  22.2   75.6  60.3  90.0  38.8 112. 
7 minute_wake    1382  56.5  11.8   55.8  48.4  63.6  38.1  78.0
# A tibble: 7 × 9
  metric            N  Mean    SD Median     Q1    Q3     P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl>  <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep  1360 102.  24.9  101.    86.1  116.   67.0  144. 
2 minute_awake   1360  13.0  9.87   9.77   6.75  15.0   3.69  35.5
3 minute_deep    1003  24.7 11.9   24.3   17     31.5   6     45.7
4 minute_in_bed  1360 117.  27.2  112.    98

In [43]:
head(insomnia_sleep_sum)
patient_stats_sum <-sleep_prep_stats(df = insomnia_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Insomnia/sleep/', cohort = 'Insomnia', main = "sum")

person_id,date,has_main_sleep,has_nap,n_records,minute_in_bed,minute_asleep,minute_awake,minute_restless,minute_deep,⋯,minute_wake,minute_after_wakeup,main_minute_in_bed,main_minute_asleep,main_minute_awake,main_minute_deep,main_minute_light,main_minute_rem,main_minute_wake,sleep_efficiency
<dbl>,<date>,<lgl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1003126,2014-02-16,TRUE,FALSE,1,541,530,11,9,0,⋯,0,0,541,530,11,0,0,0,0,97.96673
1003126,2014-02-17,TRUE,FALSE,1,489,486,3,3,0,⋯,0,0,489,486,3,0,0,0,0,99.38650
1003126,2014-02-18,TRUE,FALSE,1,461,444,17,15,0,⋯,0,0,461,444,17,0,0,0,0,96.31236
1003126,2014-02-19,TRUE,FALSE,1,508,494,14,12,0,⋯,0,0,508,494,14,0,0,0,0,97.24409
1003126,2014-02-20,TRUE,FALSE,1,505,485,20,16,0,⋯,0,0,505,485,20,0,0,0,0,96.03960
1003126,2014-02-21,TRUE,FALSE,1,510,501,9,7,0,⋯,0,0,510,501,9,0,0,0,0,98.23529


# A tibble: 8 × 9
  metric               N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>            <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep     1383 400.  66.6   406.  361.  440.  282.  505. 
2 minute_awake      1383  56.7 15.1    56.1  47.1  65.2  34.8  81.0
3 minute_deep       1383  51.8 18.3    52.7  38.8  64.7  20.7  80.3
4 minute_in_bed     1383 458.  74.4   464.  414.  503.  330.  569. 
5 minute_light      1383 236.  61.6   245.  204.  276.  125.  325. 
6 minute_rem        1383  68.9 24.9    69.6  52.4  86.1  25.6 107. 
7 minute_wake       1383  51.3 15.0    52.2  42.0  60.5  26.2  76.2
8 sleep_efficiency  1383  87.5  3.31   87.9  86.3  89.3  82.7  91.2


In [44]:
#### sleep ########
library(tidyverse)
library(bigrquery)

pca.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Prostate'),'person_id'], collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
pca_sleep_sql <- paste("
    SELECT
        sleep_daily_summary.person_id,
        sleep_daily_summary.sleep_date,
        sleep_daily_summary.is_main_sleep,
        sleep_daily_summary.minute_in_bed,
        sleep_daily_summary.minute_asleep,
        sleep_daily_summary.minute_after_wakeup,
        sleep_daily_summary.minute_awake,
        sleep_daily_summary.minute_restless,
        sleep_daily_summary.minute_deep,
        sleep_daily_summary.minute_light,
        sleep_daily_summary.minute_rem,
        sleep_daily_summary.minute_wake 
    FROM
        `sleep_daily_summary` sleep_daily_summary   
    WHERE
        PERSON_ID IN (", pca.pid.sql, ")"
)
# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
pca_sleep_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "pca_sleep",
  "pca_sleep.csv")
message(str_glue('The data will be written to {pca_sleep_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), pca_sleep_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  pca_sleep_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_sleep/pca_sleep.csv. Use this path when reading the data into your notebooks in the future.



In [45]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(is_main_sleep = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_sleep_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_sleep/pca_sleep.csv'

pca_sleep <- read_bq_export_from_workspace_bucket(pca_sleep_path)

dim(pca_sleep)

head(pca_sleep, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_sleep/pca_sleep.csv.



[1] 106337     12

person_id,sleep_date,is_main_sleep,minute_in_bed,minute_asleep,minute_after_wakeup,minute_awake,minute_restless,minute_deep,minute_light,minute_rem,minute_wake
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2673575,2023-08-23,true,190,162,4,28,NA,12,133,17,28
2673575,2023-08-22,true,208,178,0,30,NA,0,150,28,30
2673575,2023-08-05,false,245,220,0,25,NA,61,108,51,25
2673575,2023-09-07,true,264,233,0,31,NA,19,189,25,31
2673575,2023-09-29,true,326,300,0,26,NA,62,158,80,26


In [46]:
colnames(pca_sleep)[2] <- 'date'
pca_sleep_main <- pca_sleep |>
  filter(is_main_sleep == "true")

pca_sleep_nap <- pca_sleep |>
  filter(is_main_sleep == "false")

pca_sleep_sum <- sleep_sum_main_nap_func(df = pca_sleep)

In [47]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Prostate/sleep/', df = pca_sleep_sum)

── Wear compliance ──
     n_days       followup_days    
 Min.   :  23.0   Min.   :  23.00  
 1st Qu.:  70.0   1st Qu.:  82.25  
 Median :  90.0   Median :  98.00  
 Mean   : 362.4   Mean   : 383.28  
 3rd Qu.: 220.0   3rd Qu.: 239.50  
 Max.   :3466.0   Max.   :3523.00  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1015411   Min.   :  23.0   Min.   :2014-01-18   Min.   :2020-06-21  
 1st Qu.:1903888   1st Qu.:  70.0   1st Qu.:2023-01-25   1st Qu.:2023-10-01  
 Median :3428482   Median :  90.0   Median :2023-06-22   Median :2023-10-01  
 Mean   :4265862   Mean   : 362.4   Mean   :2022-08-20   Mean   :2023-09-06  
 3rd Qu.:6505875   3rd Qu.: 220.0   3rd Qu.:2023-07-04   3rd Qu.:2023-10-01  
 Max.   :9933104   Max.   :3466.0   Max.   :2023-09-09   Max.   :2023-10-01  
 followup_days      missing_days     pct_missing        compliance_tier
 Min.   :  23.00   Min.   :  0.00   Min.   : 0.000   High (>90%):164   
 1st Qu.:  82.25   1st Qu.:  3.00   1st Qu.: 2.217   80-90%     : 54   
 Median :  98.00   Median :  8.00   Median : 5.405   70-80%     : 32   
 Mean   : 383.28   Mean   : 20.89   Mean   : 8.460   60-70%     :  0   
 3rd Qu.: 239.50   3rd Qu.: 20.00   3rd Qu.:13.202   50-60%     :  0   
 Max.   :3523.00   Max

NULL

In [48]:
miss_gap_func(df = pca_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Prostate/sleep/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   0.00    1.00    2.00    4.72    7.75   14.00 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0        241         96.4          9          3.6
2         7         63         25.2        187         74.8
3        14          0          0          250        100  
4        30          0          0          250        100  
5        60          0          0          250        100  


NULL

In [49]:
patient_stats_main <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Prostate/sleep/',
    cohort = 'Prostate',
    df = pca_sleep_main,
    main = 'main'
)
patient_stats_nap <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Prostate/sleep/',
    cohort = 'Prostate',
    df = pca_sleep_nap,
    main = 'nap'
)

# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep   250 377.   66.7  383.  339.  424.  256.  466. 
2 minute_awake    250  56.5  15.2   56.0  47.9  66.1  33.3  79.0
3 minute_deep     249  47.9  14.1   47.3  38.4  56.6  27.3  72.7
4 minute_in_bed   250 434.   75.3  436.  390.  488.  305.  540. 
5 minute_light    249 276.   47.7  277.  243.  308.  203.  355. 
6 minute_rem      249  63.1  20.2   64    47.6  77.3  31.2  96.6
7 minute_wake     249  58.5  12.0   57.9  51.1  66.3  39.8  78.3
# A tibble: 7 × 9
  metric            N  Mean    SD Median     Q1    Q3     P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl>  <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep   246  95.3 24.5   95.9   79.5  108.   60.1  129. 
2 minute_awake    246  11.0  9.27   8.37   5.75  12.2   2.94  29.1
3 minute_deep     153  20.0 12.5   19     12.5   28     0     40.4
4 minute_in_bed   246 108.  25.8  105.    91

In [50]:
head(pca_sleep_sum)
patient_stats_sum <-sleep_prep_stats(df = pca_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Prostate/sleep/', cohort = 'Prostate', main = "sum")

person_id,date,has_main_sleep,has_nap,n_records,minute_in_bed,minute_asleep,minute_awake,minute_restless,minute_deep,⋯,minute_wake,minute_after_wakeup,main_minute_in_bed,main_minute_asleep,main_minute_awake,main_minute_deep,main_minute_light,main_minute_rem,main_minute_wake,sleep_efficiency
<dbl>,<date>,<lgl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1015411,2023-07-03,TRUE,FALSE,1,337,296,41,0,12,⋯,41,0,337,296,41,12,193,91,41,87.83383
1015411,2023-07-04,TRUE,FALSE,1,352,312,40,0,33,⋯,40,40,352,312,40,33,203,76,40,88.63636
1015411,2023-07-05,TRUE,FALSE,1,341,309,32,0,27,⋯,32,35,341,309,32,27,203,79,32,90.61584
1015411,2023-07-06,TRUE,FALSE,1,385,339,46,0,32,⋯,46,0,385,339,46,32,227,80,46,88.05195
1015411,2023-07-07,TRUE,FALSE,1,344,301,43,0,56,⋯,43,0,344,301,43,56,191,54,43,87.50000
1015411,2023-07-08,TRUE,FALSE,1,340,295,45,0,70,⋯,45,0,340,295,45,70,160,65,45,86.76471


# A tibble: 8 × 9
  metric               N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>            <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep      250 393.  63.0   398.  360.  437.  282.  485. 
2 minute_awake       250  58.1 15.7    57.8  48.3  66.4  34.3  81.4
3 minute_deep        250  42.7 16.5    43.2  32.0  54.7  16.8  69.8
4 minute_in_bed      250 452.  71.2   453.  413.  502.  331.  550. 
5 minute_light       250 245.  70.5   254.  216.  289.  117.  336. 
6 minute_rem         250  56.5 22.7    58.4  40.1  74.5  16.3  92.0
7 minute_wake        250  52.0 16.3    52.5  42.9  62.8  23.3  75.1
8 sleep_efficiency   250  87.2  3.10   87.5  85.9  88.9  83.0  90.6


In [51]:
#### sleep ########
library(tidyverse)
library(bigrquery)

healthy.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Healthy'),'person_id'], collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
healthy_sleep_sql <- paste("
    SELECT
        sleep_daily_summary.person_id,
        sleep_daily_summary.sleep_date,
        sleep_daily_summary.is_main_sleep,
        sleep_daily_summary.minute_in_bed,
        sleep_daily_summary.minute_asleep,
        sleep_daily_summary.minute_after_wakeup,
        sleep_daily_summary.minute_awake,
        sleep_daily_summary.minute_restless,
        sleep_daily_summary.minute_deep,
        sleep_daily_summary.minute_light,
        sleep_daily_summary.minute_rem,
        sleep_daily_summary.minute_wake 
    FROM
        `sleep_daily_summary` sleep_daily_summary   
    WHERE
        PERSON_ID IN (", healthy.pid.sql, ")"
)
# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
healthy_sleep_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "healthy_sleep",
  "healthy_sleep.csv")
message(str_glue('The data will be written to {healthy_sleep_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), healthy_sleep_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  healthy_sleep_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_sleep/healthy_sleep.csv. Use this path when reading the data into your notebooks in the future.



In [52]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(is_main_sleep = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_sleep_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_sleep/healthy_sleep.csv'

healthy_sleep <- read_bq_export_from_workspace_bucket(healthy_sleep_path)

dim(healthy_sleep)

head(healthy_sleep, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_sleep/healthy_sleep.csv.



[1] 3152699      12

person_id,sleep_date,is_main_sleep,minute_in_bed,minute_asleep,minute_after_wakeup,minute_awake,minute_restless,minute_deep,minute_light,minute_rem,minute_wake
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2909209,2023-02-02,true,403,0,0,0,NA,0,0,0,0
2909209,2023-01-31,true,464,0,0,0,NA,0,0,0,0
2880475,2023-03-28,false,182,158,4,24,NA,19,135,4,24
2867880,2018-03-28,true,183,158,6,25,NA,23,91,44,25
2907139,2023-01-22,false,180,160,0,20,NA,32,92,36,20


In [53]:
colnames(healthy_sleep)[2] <- 'date'
healthy_sleep_main <- healthy_sleep |>
  filter(is_main_sleep == "true")

healthy_sleep_nap <- healthy_sleep |>
  filter(is_main_sleep == "false")

healthy_sleep_sum <- sleep_sum_main_nap_func(df = healthy_sleep)

In [54]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Healthy/sleep/', df = healthy_sleep_sum)

── Wear compliance ──
     n_days     followup_days   
 Min.   :  21   Min.   :  21.0  
 1st Qu.:  82   1st Qu.:  91.0  
 Median : 108   Median : 121.0  
 Mean   : 434   Mean   : 459.9  
 3rd Qu.: 605   3rd Qu.: 645.0  
 Max.   :4049   Max.   :4197.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days       first_date           last_date         
 Min.   :1000464   Min.   :  21   Min.   :2012-04-05   Min.   :2016-02-05  
 1st Qu.:2251610   1st Qu.:  82   1st Qu.:2021-07-31   1st Qu.:2023-09-29  
 Median :4194375   Median : 108   Median :2023-05-20   Median :2023-10-01  
 Mean   :4719200   Mean   : 434   Mean   :2022-04-09   Mean   :2023-07-12  
 3rd Qu.:7058659   3rd Qu.: 605   3rd Qu.:2023-07-03   3rd Qu.:2023-10-01  
 Max.   :9997301   Max.   :4049   Max.   :2023-09-11   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing        compliance_tier
 Min.   :  21.0   Min.   :  0.00   Min.   : 0.000   High (>90%):4136  
 1st Qu.:  91.0   1st Qu.:  4.00   1st Qu.: 2.326   80-90%     :1548  
 Median : 121.0   Median : 14.00   Median : 5.921   70-80%     : 697  
 Mean   : 459.9   Mean   : 25.86   Mean   : 8.619   60-70%     :   0  
 3rd Qu.: 645.0   3rd Qu.: 27.00   3rd Qu.:13.710   50-60%     :   0  
 Max.   :4197.0   Max.   :459.00   Max.   

NULL

In [55]:
miss_gap_func(df = healthy_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Healthy/sleep/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   1.000   3.000   5.562  11.000  14.000 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0       6124         96.0        257         4.03
2         7       2088         32.7       4293        67.3 
3        14          0          0         6381       100   
4        30          0          0         6381       100   
5        60          0          0         6381       100   


NULL

In [56]:
patient_stats_main <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Healthy/sleep/',
    cohort = 'Healthy',
    df = healthy_sleep_main,
    main = 'main'
)
patient_stats_nap <- sleep_prep_stats(
    path.out = '/home/jupyter/workspaces/result/final/Healthy/sleep/',
    cohort = 'Healthy',
    df = healthy_sleep_nap,
    main = 'nap'
)

# A tibble: 7 × 9
  metric            N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep  6381 388.   59.1  394.  356.  426.  282.  471. 
2 minute_awake   6381  54.6  13.2   54.1  46.6  62.1  34.1  75.8
3 minute_deep    6365  59.1  16.1   59.2  47.8  70.2  32.7  85.5
4 minute_in_bed  6381 443.   66.4  451.  407.  486.  322.  536. 
5 minute_light   6365 257.   41.7  255.  229.  283.  191.  328. 
6 minute_rem     6365  78.9  20.8   79.8  64.9  93.3  43.3 112. 
7 minute_wake    6365  56.7  11.2   56.0  49.1  63.6  39.3  76.1
# A tibble: 7 × 9
  metric            N  Mean    SD Median     Q1    Q3     P5   P95
  <chr>         <int> <dbl> <dbl>  <dbl>  <dbl> <dbl>  <dbl> <dbl>
1 minute_asleep  6234 102.  26.9  100     86.1  115.   65.4  145. 
2 minute_awake   6234  12.3  9.35   9.33   6.67  13.8   3.67  34.3
3 minute_deep    4406  24.5 11.5   24.2   17.1   31.1   6     43.5
4 minute_in_bed  6234 116.  28.4  112.    97

In [57]:
head(pca_sleep_sum)
patient_stats_sum <-sleep_prep_stats(df = healthy_sleep_sum, path.out =  '/home/jupyter/workspaces/result/final/Healthy/sleep/', cohort = 'Healthy', main = "sum")

person_id,date,has_main_sleep,has_nap,n_records,minute_in_bed,minute_asleep,minute_awake,minute_restless,minute_deep,⋯,minute_wake,minute_after_wakeup,main_minute_in_bed,main_minute_asleep,main_minute_awake,main_minute_deep,main_minute_light,main_minute_rem,main_minute_wake,sleep_efficiency
<dbl>,<date>,<lgl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1015411,2023-07-03,TRUE,FALSE,1,337,296,41,0,12,⋯,41,0,337,296,41,12,193,91,41,87.83383
1015411,2023-07-04,TRUE,FALSE,1,352,312,40,0,33,⋯,40,40,352,312,40,33,203,76,40,88.63636
1015411,2023-07-05,TRUE,FALSE,1,341,309,32,0,27,⋯,32,35,341,309,32,27,203,79,32,90.61584
1015411,2023-07-06,TRUE,FALSE,1,385,339,46,0,32,⋯,46,0,385,339,46,32,227,80,46,88.05195
1015411,2023-07-07,TRUE,FALSE,1,344,301,43,0,56,⋯,43,0,344,301,43,56,191,54,43,87.50000
1015411,2023-07-08,TRUE,FALSE,1,340,295,45,0,70,⋯,45,0,340,295,45,70,160,65,45,86.76471


# A tibble: 8 × 9
  metric               N  Mean    SD Median    Q1    Q3    P5   P95
  <chr>            <int> <dbl> <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>
1 minute_asleep     6381 403.  55.4   406.  373.  437.  309.  484. 
2 minute_awake      6381  56.4 13.6    55.7  48.3  64.0  35.6  78.2
3 minute_deep       6381  54.6 18.3    55.5  42.7  67.1  22.7  83.1
4 minute_in_bed     6381 460.  62.2   464.  425.  499.  354.  551. 
5 minute_light      6381 236.  57.6   242.  209.  272.  133.  315. 
6 minute_rem        6381  72.7 23.9    74.6  57.5  89.7  29.7 109. 
7 minute_wake       6381  52.1 14.2    52.8  44.7  61    27.5  73.8
8 sleep_efficiency  6381  87.6  2.78   87.9  86.5  89.1  83.6  91.1


In [58]:
### hr
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Diabetes'),'person_id'], collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
diabetes_hr_sql <- paste("
    SELECT
        heart_rate_summary.person_id,
        heart_rate_summary.date,
        heart_rate_summary.zone_name,
        heart_rate_summary.min_heart_rate,
        heart_rate_summary.max_heart_rate,
        heart_rate_summary.minute_in_zone,
        heart_rate_summary.calorie_count 
    FROM
        `heart_rate_summary` heart_rate_summary   
    WHERE
        PERSON_ID IN (", diabetes.pid.sql, ")"
)


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
diabetes_hr_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_hr",
  "diabetes_hr.csv")
message(str_glue('The data will be written to {diabetes_hr_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_hr_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_hr_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_hr/diabetes_hr.csv. Use this path when reading the data into your notebooks in the future.



In [59]:
library(tidyverse)
library(bigrquery)

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_heart_rate_summary_98580683_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(zone_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_hr_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_hr/diabetes_hr.csv'

diabetes_hr <- read_bq_export_from_workspace_bucket(diabetes_hr_path)

dim(diabetes_hr)

head(diabetes_hr, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/diabetes_hr/diabetes_hr.csv.



[1] 1603956       7

person_id,date,zone_name,min_heart_rate,max_heart_rate,minute_in_zone,calorie_count
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
5110219,2023-08-11,Out of Range,30,108,1440,2743.482
5110219,2023-09-03,Out of Range,30,107,1438,2549.710
5110219,2023-08-03,Out of Range,30,108,1439,2859.238
5110219,2023-07-30,Out of Range,30,107,1440,2500.767
5110219,2023-09-19,Out of Range,30,109,1439,2396.346


In [60]:
diabetes_hr_unique_day <- diabetes_hr |>
  distinct(person_id, date, .keep_all = TRUE)
head(diabetes_hr_unique_day)
dim(diabetes_hr_unique_day)

person_id,date,zone_name,min_heart_rate,max_heart_rate,minute_in_zone,calorie_count
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
5110219,2023-08-11,Out of Range,30,108,1440,2743.482
5110219,2023-09-03,Out of Range,30,107,1438,2549.710
5110219,2023-08-03,Out of Range,30,108,1439,2859.238
5110219,2023-07-30,Out of Range,30,107,1440,2500.767
5110219,2023-09-19,Out of Range,30,109,1439,2396.346
5110219,2023-09-09,Out of Range,30,107,1438,2461.159


[1] 400989      7

In [61]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Diabetes/hr/', df = diabetes_hr_unique_day)

── Wear compliance ──
     n_days       followup_days   
 Min.   :  22.0   Min.   :  22.0  
 1st Qu.:  86.0   1st Qu.:  91.0  
 Median : 102.0   Median : 112.0  
 Mean   : 308.9   Mean   : 319.1  
 3rd Qu.: 277.8   3rd Qu.: 280.8  
 Max.   :2931.0   Max.   :2970.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1003770   Min.   :  22.0   Min.   :2015-06-22   Min.   :2018-07-09  
 1st Qu.:1940734   1st Qu.:  86.0   1st Qu.:2022-11-25   1st Qu.:2023-10-01  
 Median :3361450   Median : 102.0   Median :2023-06-11   Median :2023-10-01  
 Mean   :4321604   Mean   : 308.9   Mean   :2022-09-21   Mean   :2023-08-05  
 3rd Qu.:6542119   3rd Qu.: 277.8   3rd Qu.:2023-07-03   3rd Qu.:2023-10-01  
 Max.   :9987610   Max.   :2931.0   Max.   :2023-09-10   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing        compliance_tier
 Min.   :  22.0   Min.   :  0.00   Min.   : 0.000   High (>90%):1042  
 1st Qu.:  91.0   1st Qu.:  0.00   1st Qu.: 0.000   80-90%     : 220  
 Median : 112.0   Median :  3.00   Median : 2.913   70-80%     :  36  
 Mean   : 319.1   Mean   : 10.17   Mean   : 5.026   60-70%     :   0  
 3rd Qu.: 280.8   3rd Qu.: 14.00   3rd Qu.: 7.368   50-60%     :   0  
 Max.   :2970.0   Max.   :20

NULL

In [62]:
miss_gap_func(df = diabetes_hr_unique_day, path.out =  '/home/jupyter/workspaces/result/final/Diabetes/hr/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.000   2.000   3.833   5.000  14.000 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0        926         71.3        372         28.7
2         7        275         21.2       1023         78.8
3        14          0          0         1298        100  
4        30          0          0         1298        100  
5        60          0          0         1298        100  


NULL

In [63]:
### hr
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Insomnia'),'person_id'], collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
insomnia_hr_sql <- paste("
    SELECT
        heart_rate_summary.person_id,
        heart_rate_summary.date,
        heart_rate_summary.zone_name,
        heart_rate_summary.min_heart_rate,
        heart_rate_summary.max_heart_rate,
        heart_rate_summary.minute_in_zone,
        heart_rate_summary.calorie_count 
    FROM
        `heart_rate_summary` heart_rate_summary   
    WHERE
        PERSON_ID IN (", insomnia.pid.sql, ")"
)


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
insomnia_hr_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_hr",
  "insomnia_hr.csv")
message(str_glue('The data will be written to {insomnia_hr_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_hr_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_hr_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_hr/insomnia_hr.csv. Use this path when reading the data into your notebooks in the future.



In [64]:
library(tidyverse)
library(bigrquery)

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_heart_rate_summary_98580683_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(zone_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_hr_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_hr/insomnia_hr.csv'

insomnia_hr <- read_bq_export_from_workspace_bucket(insomnia_hr_path)

dim(insomnia_hr)

head(insomnia_hr, 3)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/insomnia_hr/insomnia_hr.csv.



[1] 1887904       7

person_id,date,zone_name,min_heart_rate,max_heart_rate,minute_in_zone,calorie_count
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1887684,2023-09-05,Out of Range,30,108,1407,1298.625
1887684,2023-08-18,Out of Range,30,108,1414,1487.991
1887684,2023-09-06,Out of Range,30,108,1440,1400.356


In [65]:
insomnia_hr_unique_day <- insomnia_hr |>
  distinct(person_id, date, .keep_all = TRUE)
head(insomnia_hr_unique_day)
dim(insomnia_hr_unique_day)

person_id,date,zone_name,min_heart_rate,max_heart_rate,minute_in_zone,calorie_count
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1887684,2023-09-05,Out of Range,30,108,1407,1298.625
1887684,2023-08-18,Out of Range,30,108,1414,1487.991
1887684,2023-09-06,Out of Range,30,108,1440,1400.356
1887684,2023-09-07,Out of Range,30,107,1434,1327.889
1887684,2023-09-13,Out of Range,30,106,1404,1474.266
1887684,2023-07-31,Out of Range,30,105,1434,1340.676


[1] 471976      7

In [66]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Insomnia/hr/', df = insomnia_hr_unique_day)

── Wear compliance ──
     n_days       followup_days   
 Min.   :  22.0   Min.   :  22.0  
 1st Qu.:  84.0   1st Qu.:  91.0  
 Median : 101.0   Median : 111.0  
 Mean   : 341.3   Mean   : 353.1  
 3rd Qu.: 294.0   3rd Qu.: 300.5  
 Max.   :5387.0   Max.   :5387.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1003126   Min.   :  22.0   Min.   :2009-01-01   Min.   :2017-11-21  
 1st Qu.:1870796   1st Qu.:  84.0   1st Qu.:2022-10-01   1st Qu.:2023-10-01  
 Median :3149420   Median : 101.0   Median :2023-06-12   Median :2023-10-01  
 Mean   :4182739   Mean   : 341.3   Mean   :2022-08-11   Mean   :2023-07-30  
 3rd Qu.:6451714   3rd Qu.: 294.0   3rd Qu.:2023-07-03   3rd Qu.:2023-10-01  
 Max.   :9987114   Max.   :5387.0   Max.   :2023-09-10   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing        compliance_tier
 Min.   :  22.0   Min.   :  0.00   Min.   : 0.000   High (>90%):1106  
 1st Qu.:  91.0   1st Qu.:  0.00   1st Qu.: 0.000   80-90%     : 242  
 Median : 111.0   Median :  4.00   Median : 3.058   70-80%     :  35  
 Mean   : 353.1   Mean   : 11.84   Mean   : 5.095   60-70%     :   0  
 3rd Qu.: 300.5   3rd Qu.: 14.50   3rd Qu.: 7.785   50-60%     :   0  
 Max.   :5387.0   Max.   :19

NULL

In [67]:
miss_gap_func(df = insomnia_hr_unique_day, path.out =  '/home/jupyter/workspaces/result/final/Insomnia/hr/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.000   2.000   3.867   5.000  14.000 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0       1006         72.7        377         27.3
2         7        291         21.0       1092         79.0
3        14          0          0         1383        100  
4        30          0          0         1383        100  
5        60          0          0         1383        100  


NULL

In [68]:
### hr
library(tidyverse)
library(bigrquery)

pca.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Prostate'),'person_id'], collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
pca_hr_sql <- paste("
    SELECT
        heart_rate_summary.person_id,
        heart_rate_summary.date,
        heart_rate_summary.zone_name,
        heart_rate_summary.min_heart_rate,
        heart_rate_summary.max_heart_rate,
        heart_rate_summary.minute_in_zone,
        heart_rate_summary.calorie_count 
    FROM
        `heart_rate_summary` heart_rate_summary   
    WHERE
        PERSON_ID IN (", pca.pid.sql, ")"
)


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
pca_hr_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "pca_hr",
  "pca_hr.csv")
message(str_glue('The data will be written to {pca_hr_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), pca_hr_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  pca_hr_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_hr/pca_hr.csv. Use this path when reading the data into your notebooks in the future.



In [69]:
library(tidyverse)
library(bigrquery)

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_heart_rate_summary_98580683_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(zone_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_hr_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_hr/pca_hr.csv'

pca_hr <- read_bq_export_from_workspace_bucket(pca_hr_path)

dim(pca_hr)

head(pca_hr, 3)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/pca_hr/pca_hr.csv.



[1] 338364      7

person_id,date,zone_name,min_heart_rate,max_heart_rate,minute_in_zone,calorie_count
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
8833613,2023-08-30,Out of Range,30,92,1430,2789.039
8833613,2023-09-20,Out of Range,30,96,1414,2853.363
8833613,2023-08-22,Out of Range,30,93,1414,2549.952


In [70]:
pca_hr_unique_day <- pca_hr |>
  distinct(person_id, date, .keep_all = TRUE)
head(pca_hr_unique_day)
dim(pca_hr_unique_day)

person_id,date,zone_name,min_heart_rate,max_heart_rate,minute_in_zone,calorie_count
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
8833613,2023-08-30,Out of Range,30,92,1430,2789.039
8833613,2023-09-20,Out of Range,30,96,1414,2853.363
8833613,2023-08-22,Out of Range,30,93,1414,2549.952
8833613,2023-07-07,Out of Range,30,96,1389,2876.327
8833613,2023-08-10,Out of Range,30,94,1394,3560.391
8833613,2023-09-18,Out of Range,30,94,1437,2934.299


[1] 84591     7

In [71]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Prostate/hr/', df = pca_hr_unique_day)

── Wear compliance ──
     n_days       followup_days   
 Min.   :  23.0   Min.   :  23.0  
 1st Qu.:  79.0   1st Qu.:  84.0  
 Median :  93.5   Median :  99.5  
 Mean   : 338.4   Mean   : 350.2  
 3rd Qu.: 222.8   3rd Qu.: 239.0  
 Max.   :3081.0   Max.   :3172.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1015411   Min.   :  23.0   Min.   :2015-01-25   Min.   :2020-06-21  
 1st Qu.:1903888   1st Qu.:  79.0   1st Qu.:2023-02-05   1st Qu.:2023-10-01  
 Median :3428482   Median :  93.5   Median :2023-06-21   Median :2023-10-01  
 Mean   :4265862   Mean   : 338.4   Mean   :2022-09-26   Mean   :2023-09-10  
 3rd Qu.:6505875   3rd Qu.: 222.8   3rd Qu.:2023-07-04   3rd Qu.:2023-10-01  
 Max.   :9933104   Max.   :3081.0   Max.   :2023-09-09   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing         compliance_tier
 Min.   :  23.0   Min.   :  0.00   Min.   : 0.0000   High (>90%):209   
 1st Qu.:  84.0   1st Qu.:  1.00   1st Qu.: 0.9792   80-90%     : 38   
 Median :  99.5   Median :  3.00   Median : 3.0442   70-80%     :  3   
 Mean   : 350.2   Mean   : 11.88   Mean   : 4.7314   60-70%     :  0   
 3rd Qu.: 239.0   3rd Qu.: 13.75   3rd Qu.: 6.7346   50-60%     :  0   
 Max.   :3172.0   Max.

NULL

In [72]:
miss_gap_func(df = pca_hr_unique_day, path.out =  '/home/jupyter/workspaces/result/final/Prostate/hr/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   1.000   2.000   3.508   4.000  14.000 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0        188         75.2         62         24.8
2         7         45         18          205         82  
3        14          0          0          250        100  
4        30          0          0          250        100  
5        60          0          0          250        100  


NULL

In [73]:
### hr
library(tidyverse)
library(bigrquery)

healthy.pid.sql <- paste(final_pid[which(final_pid$cohort == 'Healthy'),'person_id'], collapse = ", ")
# This query represents dataset "diabetes" for domain "fitbit_sleep_daily_summary" and was generated for All of Us Controlled Tier Dataset v8
healthy_hr_sql <- paste("
    SELECT
        heart_rate_summary.person_id,
        heart_rate_summary.date,
        heart_rate_summary.zone_name,
        heart_rate_summary.min_heart_rate,
        heart_rate_summary.max_heart_rate,
        heart_rate_summary.minute_in_zone,
        heart_rate_summary.calorie_count 
    FROM
        `heart_rate_summary` heart_rate_summary   
    WHERE
        PERSON_ID IN (", healthy.pid.sql, ")"
)


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
healthy_hr_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "healthy_hr",
  "healthy_hr.csv")
message(str_glue('The data will be written to {healthy_hr_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), healthy_hr_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  healthy_hr_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_hr/healthy_hr.csv. Use this path when reading the data into your notebooks in the future.



In [74]:
library(tidyverse)
library(bigrquery)

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {fitbit_heart_rate_summary_98580683_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(zone_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_hr_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_hr/healthy_hr.csv'

healthy_hr <- read_bq_export_from_workspace_bucket(healthy_hr_path)

dim(healthy_hr)

head(healthy_hr, 3)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260511/healthy_hr/healthy_hr.csv.



[1] 10547720        7

person_id,date,zone_name,min_heart_rate,max_heart_rate,minute_in_zone,calorie_count
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
2078603,2023-08-22,Out of Range,30,108,1435,2217.602
2078603,2023-09-16,Out of Range,30,107,1427,2137.246
2078603,2023-09-01,Out of Range,30,107,1437,2043.917


In [75]:
healthy_hr_unique_day <- healthy_hr |>
  distinct(person_id, date, .keep_all = TRUE)
head(healthy_hr_unique_day)
dim(healthy_hr_unique_day)

person_id,date,zone_name,min_heart_rate,max_heart_rate,minute_in_zone,calorie_count
<dbl>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
2078603,2023-08-22,Out of Range,30,108,1435,2217.602
2078603,2023-09-16,Out of Range,30,107,1427,2137.246
2078603,2023-09-01,Out of Range,30,107,1437,2043.917
2078603,2023-08-29,Out of Range,30,104,1427,2266.344
2078603,2023-07-21,Out of Range,30,107,1412,2113.463
2078603,2023-08-14,Out of Range,30,104,1439,2191.782


[1] 2636930       7

In [76]:
wear_compliance_func(path.out = '/home/jupyter/workspaces/result/final/Healthy/hr/', df = healthy_hr_unique_day)

── Wear compliance ──
     n_days       followup_days   
 Min.   :  21.0   Min.   :  21.0  
 1st Qu.:  88.0   1st Qu.:  91.0  
 Median : 121.0   Median : 124.0  
 Mean   : 413.2   Mean   : 424.3  
 3rd Qu.: 617.0   3rd Qu.: 635.0  
 Max.   :3175.0   Max.   :3197.0  


`geom_smooth()` using formula = 'y ~ x'


   person_id           n_days         first_date           last_date         
 Min.   :1000464   Min.   :  21.0   Min.   :2014-12-12   Min.   :2016-02-05  
 1st Qu.:2251610   1st Qu.:  88.0   1st Qu.:2021-09-22   1st Qu.:2023-10-01  
 Median :4194375   Median : 121.0   Median :2023-05-23   Median :2023-10-01  
 Mean   :4719200   Mean   : 413.2   Mean   :2022-06-01   Mean   :2023-07-29  
 3rd Qu.:7058659   3rd Qu.: 617.0   3rd Qu.:2023-06-30   3rd Qu.:2023-10-01  
 Max.   :9997301   Max.   :3175.0   Max.   :2023-09-11   Max.   :2023-10-01  
 followup_days     missing_days     pct_missing        compliance_tier
 Min.   :  21.0   Min.   :  0.00   Min.   : 0.000   High (>90%):5277  
 1st Qu.:  91.0   1st Qu.:  0.00   1st Qu.: 0.000   80-90%     : 996  
 Median : 124.0   Median :  3.00   Median : 2.020   70-80%     : 108  
 Mean   : 424.3   Mean   : 11.06   Mean   : 4.271   60-70%     :   0  
 3rd Qu.: 635.0   3rd Qu.: 15.00   3rd Qu.: 6.283   50-60%     :   0  
 Max.   :3197.0   Max.   :38

NULL

In [77]:
miss_gap_func(df = healthy_hr_unique_day, path.out =  '/home/jupyter/workspaces/result/final/Healthy/hr/')

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.000   1.000   3.891   6.000  14.000 

── Exclusion sensitivity by threshold ──
# A tibble: 5 × 5
  threshold n_excluded pct_excluded n_retained pct_retained
      <dbl>      <int>        <dbl>      <int>        <dbl>
1         0       4013         62.9       2368         37.1
2         7       1515         23.7       4866         76.3
3        14          0          0         6381        100  
4        30          0          0         6381        100  
5        60          0          0         6381        100  


NULL

In [78]:
activity = rbind(diabetes_activity, insomnia_activity, pca_activity, healthy_activity)
sleep = rbind(diabetes_sleep, insomnia_sleep, pca_sleep, healthy_sleep)
hr = rbind(diabetes_hr, insomnia_hr, pca_hr, healthy_hr)

In [79]:
dim(activity)
dim(sleep)
dim(hr)

[1] 4413932      13

[1] 4340404      12

[1] 14377944        7

In [80]:
write.table(activity, "/home/jupyter/workspaces/result/final/classifier/final_all_activity.txt", sep = '\t', quote = F, row.names = F)
write.table(sleep, "/home/jupyter/workspaces/result/final/classifier/final_all_sleep.txt", sep = '\t', quote = F, row.names = F)
write.table(hr, "/home/jupyter/workspaces/result/final/classifier/final_all_hr.txt", sep = '\t', quote = F, row.names = F)
